# Embeddings and Vec2Text inversion demo

This Colab notebook demonstrates:

1. How text becomes a numeric embedding vector.
2. How semantic similarity works with embeddings.
3. How vector search / retrieval works at a small scale.
4. How **Vec2Text** can attempt to reconstruct text from embeddings.
5. Why embeddings should be treated as sensitive derived data.

> Recommended runtime: **GPU**  
> In Colab: `Runtime` → `Change runtime type` → `T4 GPU` or similar.


## 0. Install dependencies

This notebook uses two paths:

- **Local/free path:** `sentence-transformers/gtr-t5-base` + `vec2text` for embedding inversion.
- **Optional OpenAI path:** OpenAI embedding API for modern production-style embeddings.

Vec2Text inversion works only when the inversion model matches the embedding model. The local demo below uses a GTR-compatible embedding model and corrector.


In [1]:
# Colab dependency setup.
# Vec2Text is currently sensitive to newer Transformers releases.
# Pin Transformers below 4.50 because vec2text has a known compatibility issue with Transformers 4.50+.

!pip -q install --upgrade pip
!pip -q install 'transformers==4.49.0' 'accelerate<1.8' vec2text sentencepiece numpy pandas scikit-learn matplotlib openai

# After this cell finishes in Colab, use Runtime -> Restart runtime, then continue from the next cell.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 37.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import os
import math
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from sklearn.metrics.pairwise import cosine_similarity

# Defensive fix for notebooks where some prior code/library left PyTorch's default device as 'meta'.
# Transformers cannot load pretrained weights while the default device is meta.
try:
    torch.set_default_device('cpu')
except Exception as e:
    print('Could not reset default device; continuing:', repr(e))

device = 'cuda' if torch.cuda.is_available() else 'cpu'
device


'cuda'

## 1. Create embeddings locally

An embedding is a vector of floating-point numbers. Similar meanings should produce vectors that are close together.

Here we use `sentence-transformers/gtr-t5-base`, because Vec2Text has examples for inverting GTR-style embeddings.


In [3]:
import vec2text
from transformers import AutoModel, AutoTokenizer, PreTrainedModel, PreTrainedTokenizer

encoder_name = "sentence-transformers/gtr-t5-base"

tokenizer = AutoTokenizer.from_pretrained(encoder_name)
encoder = AutoModel.from_pretrained(encoder_name).encoder.to(device)
encoder.eval()

def get_gtr_embeddings(
    text_list,
    encoder: PreTrainedModel = encoder,
    tokenizer: PreTrainedTokenizer = tokenizer,
    max_length: int = 128,
) -> torch.Tensor:
    """Create GTR embeddings compatible with the Vec2Text gtr-base corrector."""
    inputs = tokenizer(
        text_list,
        return_tensors="pt",
        max_length=max_length,
        truncation=True,
        padding="max_length",
    ).to(device)

    with torch.no_grad():
        model_output = encoder(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
        )
        hidden_state = model_output.last_hidden_state
        embeddings = vec2text.models.model_utils.mean_pool(
            hidden_state,
            inputs["attention_mask"],
        )

    return embeddings

texts = [
    "How do I reset my password?",
    "I forgot my password and cannot access my account.",
    "The Azure Front Door managed certificate failed to renew.",
    "A cat is sleeping on the sofa.",
    "The kitten is resting on the couch.",
]

emb = get_gtr_embeddings(texts)

print("Embedding tensor shape:", tuple(emb.shape))
print("One embedding vector length:", emb.shape[1])
print("First 10 values of first embedding:")
print(emb[0, :10].detach().cpu().numpy())


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/219M [00:00<?, ?B/s]

Embedding tensor shape: (5, 768)
One embedding vector length: 768
First 10 values of first embedding:
[ 0.12280415 -0.0386237  -0.00719023 -0.05895328  0.01107384  0.02566082
  0.01434232 -0.04613852  0.01460797  0.10543957]


## 2. Compare semantic similarity

Cosine similarity is commonly used to compare vectors.

Values closer to `1.0` usually mean the vectors are more similar.


In [4]:
emb_np = emb.detach().cpu().numpy()
sim = cosine_similarity(emb_np)

df = pd.DataFrame(sim, index=texts, columns=texts)
df


,How do I reset my password?,I forgot my password and cannot access my account.,The Azure Front Door managed certificate failed to renew.,A cat is sleeping on the sofa.,The kitten is resting on the couch.
How do I reset my password?,1.000000,0.546802,0.174421,0.033601,0.038375
I forgot my password and cannot access my account.,0.546802,1.000000,0.266765,0.108443,0.108666
The Azure Front Door managed certificate failed to renew.,0.174421,0.266765,1.000000,0.168853,0.141056
A cat is sleeping on the sofa.,0.033601,0.108443,0.168853,1.000000,0.762473
The kitten is resting on the couch.,0.038375,0.108666,0.141056,0.762473,1.000000


In [5]:
# Query-like example: rank documents by semantic similarity.

documents = [
    "Password reset instructions for users.",
    "OIDC login troubleshooting guide.",
    "Azure Front Door custom domain and certificate renewal.",
    "How to groom a cat.",
    "SharePoint Server installation guide.",
]

query = "My user cannot sign in and needs account recovery."

doc_emb = get_gtr_embeddings(documents)
query_emb = get_gtr_embeddings([query])

scores = cosine_similarity(
    query_emb.detach().cpu().numpy(),
    doc_emb.detach().cpu().numpy(),
)[0]

ranking = pd.DataFrame({
    "document": documents,
    "similarity": scores,
}).sort_values("similarity", ascending=False)

ranking


,document,similarity
0,Password reset instructions for users.,0.552850
1,OIDC login troubleshooting guide.,0.342736
2,Azure Front Door custom domain and certificate...,0.180891
4,SharePoint Server installation guide.,0.166818
3,How to groom a cat.,0.135847


## 3. Optional: OpenAI embeddings

This section demonstrates production-style OpenAI embeddings. It is optional and requires an API key.

Important: the Vec2Text GTR demo below does **not** invert OpenAI `text-embedding-3-small` vectors. Inversion requires an inversion/corrector model trained for the same embedding space.

OpenAI currently documents `text-embedding-3-small` as 1536 dimensions by default and `text-embedding-3-large` as 3072 dimensions by default.


In [6]:
# Optional: run only if you have an OpenAI API key.
# In Colab, you can either:
# 1. paste it temporarily with getpass, or
# 2. set it in Colab Secrets and read it from os.environ.

RUN_OPENAI_DEMO = False

if RUN_OPENAI_DEMO:
    from getpass import getpass
    from openai import OpenAI

    if not os.environ.get("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")

    client = OpenAI()

    openai_texts = [
        "How do I reset my password?",
        "I forgot my password and cannot access my account.",
        "The Azure Front Door managed certificate failed to renew.",
    ]

    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=openai_texts,
    )

    openai_embeddings = np.array([item.embedding for item in response.data])
    print("OpenAI embedding shape:", openai_embeddings.shape)
    print("First 10 values:", openai_embeddings[0, :10])
else:
    print("OpenAI demo skipped. Set RUN_OPENAI_DEMO = True to run it.")


OpenAI demo skipped. Set RUN_OPENAI_DEMO = True to run it.


## 4. Invert text using Vec2Text

Now we attempt to reconstruct text from embeddings.

Vec2Text is not a general decoder for every embedding model. The inversion model must be compatible with the embedding model used to create the vectors.

This demo uses:

- embedding model: `sentence-transformers/gtr-t5-base`
- Vec2Text corrector: `gtr-base`

For best results, use GPU and keep examples reasonably short.


In [7]:
# Load the Vec2Text corrector.
# This downloads pretrained weights from Hugging Face.
#
# If you previously hit:
# RuntimeError: from_pretrained with a meta device context manager / torch.set_default_device('meta')
# run Runtime -> Restart runtime and execute the notebook from the top with the pinned install cell.

import torch

try:
    torch.set_default_device('cpu')
except Exception as e:
    print('Could not reset default device:', repr(e))

corrector = vec2text.load_pretrained_corrector('gtr-base')
print('Loaded corrector:', type(corrector))


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin.index.json: 0.00B [00:00, ?B/s]

pytorch_model-00001-of-00008.bin:   0%|          | 0.00/193M [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

pytorch_model-00002-of-00008.bin:   0%|          | 0.00/198M [00:00<?, ?B/s]

...

pytorch_model-00005-of-00006.bin:   0%|          | 0.00/187M [00:00<?, ?B/s]

pytorch_model-00006-of-00006.bin:   0%|          | 0.00/37.8M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/69.0 [00:00<?, ?B/s]

Loaded corrector: <class 'vec2text.trainers.corrector.Corrector'>


In [13]:
secret_texts = [
    "Alice Johnson's appointment is scheduled for Tuesday at 3 PM.",
    "The VPN gateway private key must never be committed to Git.",
    "Jack Morris is a PhD student at Cornell Tech in New York City.",
    "The API key to use is MyFancy@P1K3y!",
    "Production Tenant Id is: 10fe570d-864a-47c9-a87c-08634011a094",
    "Passphrase: Stingy-Wackiness-Feminism-Stowaway-Distress",
    "Admin password: AYskrDjJkc7wwR"
]

secret_embeddings = get_gtr_embeddings(secret_texts)

# Direct inversion from embeddings.
# num_steps improves correction quality; sequence_beam_width improves search quality but costs more GPU memory.
reconstructed = vec2text.invert_embeddings(
    embeddings=secret_embeddings.to(device),
    corrector=corrector,
    num_steps=20,
    sequence_beam_width=4,
)

pd.DataFrame({
    "original": secret_texts,
    "reconstructed": reconstructed,
})


,original,reconstructed
0,Alice Johnson's appointment is scheduled for T...,Alice Johnson's appointment is schedule...
1,The VPN gateway private key must never be comm...,The VPN gateway private key must neve...
2,Jack Morris is a PhD student at Cornell Tech i...,Jack Morris is a PhD student at Cornell ...
3,The API key to use is MyFancy@P1K3y!,The API key to use is MyFancy@p3Kvy!
4,Production Tenant Id is: 10fe570d-864a-47c9-a8...,Production Tenant Id is 0570-048-048-a-fe10-a-...
5,Passphrase: Stingy-Wackiness-Feminism-Stowaway...,Passphrase: Stingy-Wackiness-Feminism-Stowaway...
6,Admin password: AYskrDjJkc7wwR,Admin password: AYDKRj7cwrjsrj


## 5. Try different attack settings

Lower settings are faster but less accurate. Higher settings are slower and may require more GPU memory.


In [9]:
test_text = [
    "The customer reported an OIDC login failure after certificate rotation."
]

test_embedding = get_gtr_embeddings(test_text)

settings = [
    {"num_steps": 0, "sequence_beam_width": 0},
    {"num_steps": 5, "sequence_beam_width": 1},
    {"num_steps": 10, "sequence_beam_width": 2},
    {"num_steps": 20, "sequence_beam_width": 4},
]

rows = []

for cfg in settings:
    try:
        recovered = vec2text.invert_embeddings(
            embeddings=test_embedding.to(device),
            corrector=corrector,
            num_steps=cfg["num_steps"],
            sequence_beam_width=cfg["sequence_beam_width"],
        )[0]
    except RuntimeError as e:
        recovered = f"RuntimeError, possibly out of GPU memory: {e}"

    rows.append({
        "num_steps": cfg["num_steps"],
        "sequence_beam_width": cfg["sequence_beam_width"],
        "original": test_text[0],
        "reconstructed": recovered,
    })

pd.DataFrame(rows)


,num_steps,sequence_beam_width,original,reconstructed
0,0,0,The customer reported an OIDC login failure af...,customer reported OIDC after the authe...
1,5,1,The customer reported an OIDC login failure af...,The customer reported an OIDC login f...
2,10,2,The customer reported an OIDC login failure af...,The customer reported an OIDC login f...
3,20,4,The customer reported an OIDC login failure af...,The customer reported an OIDC login f...


## 6. Invert interpolated embeddings

This shows that the vector space is semantic. If we average two embeddings and invert the result, the reconstructed text may blend concepts from both originals.


In [10]:
text_a = "The user forgot their password and cannot access the account."
text_b = "The kitten is sleeping on the sofa near the window."

pair_embeddings = get_gtr_embeddings([text_a, text_b])

rows = []

for alpha in np.linspace(0, 1, 6):
    mixed = torch.lerp(pair_embeddings[0], pair_embeddings[1], float(alpha))[None, :]
    recovered = vec2text.invert_embeddings(
        embeddings=mixed.to(device),
        corrector=corrector,
        num_steps=10,
        sequence_beam_width=2,
    )[0]
    rows.append({
        "alpha": round(float(alpha), 2),
        "reconstructed_from_mixed_embedding": recovered,
    })

pd.DataFrame(rows)


,alpha,reconstructed_from_mixed_embedding
0,0.0,The user forgot their password and c...
1,0.2,The user forgot their password and is ...
2,0.4,The user forgot the password and is not able t...
3,0.6,The user cannot access the account. The kitten...
4,0.8,The kitten is sleeping on the sofa next to the...
5,1.0,The kitten is sleeping on the sofa ne...


## 7. Privacy and security takeaway

Embeddings are not reversible encodings like Base64, and they are not encryption. But embedding inversion shows that embeddings can leak substantial information about the original text.

Treat the following as sensitive:

- raw documents
- chunks
- embeddings
- vector database rows
- metadata used for retrieval

Recommended controls:

- Do not embed secrets, passwords, tokens, private keys, or unnecessary PII.
- Enforce tenant/user authorization before vector retrieval.
- Encrypt vector storage.
- Avoid exposing raw embeddings to clients.
- Delete embeddings when source documents are deleted.
- Audit vector database access.
- Use metadata filters and access-control filters before semantic ranking.


## References

- Vec2Text GitHub: https://github.com/vec2text/vec2text
- Paper: *Text Embeddings Reveal (Almost) As Much As Text*, EMNLP 2023
- Paper: *Language Model Inversion*, ICLR 2024
- OpenAI embeddings guide: https://developers.openai.com/api/docs/guides/embeddings
